In [0]:
import json
import urllib.request
from pyspark.sql import functions as F

In [0]:
# Dynamically resolve current user to form a valid Unity Catalog Workspace path
current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")
landing_path = f"{base_path}/landing"

# Fetch live seismic snapshot from USGS API (No API key required)
usgs_url = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&minmagnitude=2.5"
req = urllib.request.Request(usgs_url, headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(req) as response:
    payload = json.loads(response.read().decode())

# Flatten GeoJSON features array into tabular records
features = payload.get("features", [])[:100]
records = []
for f in features:
    props = f["properties"]
    coords = f["geometry"]["coordinates"]
    records.append({
        "event_id": str(f["id"]),
        "title": str(props.get("title")),
        "magnitude": float(props.get("mag")) if props.get("mag") is not None else 0.0,
        "place": str(props.get("place")),
        "event_time": int(props.get("time")),
        "longitude": float(coords[0]),
        "latitude": float(coords[1]),
        "depth_km": float(coords[2]),
        "tsunami_flag": int(props.get("tsunami", 0))
    })

# Batch 1: Baseline Dataset (Live API snapshot T0)
# - Written to Workspace filesystem path to comply with Unity Catalog access rules
df_b1 = spark.createDataFrame(records)
df_b1.write.mode("overwrite").json(f"{landing_path}/batch_1")

# Batch 2: Updated Telemetry (T1) + New Schema Column
# - Updates magnitude and depth_km for first 10 events to test SCD1/SCD2 MERGE logic
# - Adds AFTERSHOCK_RISK_SCORE to test Delta Schema Evolution (autoMerge)
min_event_id = df_b1.select(F.min("event_id")).collect()[0][0]

df_b2 = df_b1.withColumn(
    "magnitude",
    F.when(F.col("event_id") <= min_event_id, F.col("magnitude") + 0.5)
     .otherwise(F.col("magnitude"))
).withColumn(
    "depth_km",
    F.when(F.col("event_id") <= min_event_id, F.col("depth_km") + 2.1)
     .otherwise(F.col("depth_km"))
).withColumn(
    "event_time", F.col("event_time") + 3600000
).withColumn(
    "AFTERSHOCK_RISK_SCORE", F.round(F.rand() * 10, 2)
)
df_b2.write.mode("overwrite").json(f"{landing_path}/batch_2")

# Batch 3: Corrupted Stream (T2) for Governance Testing
# - Injects string into numeric field magnitude to test Schema Enforcement / Bad Record Handling
# - Injects UNAUTHORIZED_SENSOR_DATA to test Data Contract rejection
df_b3 = df_b1.limit(10).withColumn(
    "magnitude", F.lit("CORRUPTED_VALUE")
).withColumn(
    "UNAUTHORIZED_SENSOR_DATA", F.lit("MALICIOUS_INPUT")
)
df_b3.write.mode("overwrite").json(f"{landing_path}/batch_3")

In [0]:
# current_user = spark.sql("SELECT current_user()").collect()[0][0]
# default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

# dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
# base_path = dbutils.widgets.get("base_path")

# for batch_num in [1, 2, 3]:
#     path = f"{base_path}/landing/batch_{batch_num}"
#     df = spark.read.json(path)
    
#     print(f"=== BATCH {batch_num} METADATA & SAMPLE ===")
#     print(f"Record Count: {df.count()}")
#     print("Schema Columns:", df.columns)
    
#     eval_cols = [c for c in df.columns if c in ["AFTERSHOCK_RISK_SCORE", "UNAUTHORIZED_SENSOR_DATA", "magnitude"]]
#     df.select("event_id", "place", "event_time", *eval_cols).show(3, truncate=False)
#     print("-" * 80)